# ĐỒ ÁN TOÁN ỨNG DỤNG VÀ THỐNG KÊ
# PHÉP KHỬ GAUSS VÀ CÁC ỨNG DỤNG

- **Nhóm thực hiện:** Nhóm 14 - Lớp 24CTT2
- **Mục tiêu:** Cài đặt phép khử Gauss có partial pivoting từ đầu bằng Python để giải hệ phương trình tuyến tính, tính định thức, tìm ma trận nghịch đảo, tính hạng và tìm cơ sở.

# Thiết lập cấu hình (config)
**Khai báo thư viện, hằng số EPSILON, Import các hàm thuật toán và thêm các hàm bổ trợ**

In [ ]:
import sys, os
import pandas as pd
from IPython.display import display

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
if current_dir not in sys.path:
    sys.path.append(current_dir)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from config import EPSILON, is_zero, make_zero, AutoTestReporter

from gaussian import (gaussian_eliminate,verify_test_back_substitution, verify_test_gaussian_eliminate)
from determinant import (determinant, verify_test_determinant)
from inverse import (inverse, verify_test_inverse)
from rank_basis import (rank_and_basis, verify_test_rank_and_basis)
from test_case import (
    BACK_SUBSTITUTION_TEST_CASES,
    DETERMINANT_TEST_CASES,
    GAUSSIAN_ELIMINATE_TEST_CASES,
    INVERSE_TEST_CASES,
    RANK_BASIS_TEST_CASES,
    VERIFY_SOLUTION_TEST_CASES,

)

from verification import (verify_solution, verify_test_verify_solution,
    verify_determinant_numpy,  
    verify_inverse_numpy, 
    verify_rank_and_basis_numpy)


def display_matrix(matrix, title="Matrix", cmap="coolwarm"):
    """
    Sử dụng heatmap và pandas để hiển thị ma trận
    """
    print(f"\n--- {title} ---")
    
    if matrix is None or not matrix:
        print("   (Không tồn tại)")
        return
        
    df = pd.DataFrame(matrix)
    
    # 3. Xử lý riêng cho Vector 1 chiều: Ép thành 1 hàng ngang cho gọn
    if isinstance(matrix, list) and not isinstance(matrix[0], list):
        df = pd.DataFrame([matrix])
    
    # 4. Khử sai số float: Đưa các số cực nhỏ (gần 0) về hẳn 0.0 để bảng sạch sẽ
    df = df.map(lambda x: 0.0 if abs(x) < EPSILON else x) 
    
    styled_df = df.style.background_gradient(cmap=cmap, axis=None) \
                        .format("{:.4f}") \
                        .set_properties(**{
                            'text-align': 'center', 
                            'border': '1px solid #bbbbbb',
                            'padding': '10px'
                        })
    
    display(styled_df)
    
print(f"EPSILON có giá trị là: {EPSILON}")

# Phần 1: Demo các thuật toán

**1. Giải hệ phương trình**

In [ ]:

print("\nHỆ CÓ NGHIỆM DUY NHẤT")
A1 = [[1, 2, 3], 
     [0, 1, 4], 
     [5, 6, 0]]
b1 = [14, 32, 50]

display_matrix(A1, "Ma trận hệ số A")
display_matrix(b1, "Vector vế phải b")
Ab1_init = [row + [b1[i]] for i, row in enumerate(A1)]
display_matrix(Ab1_init, "Ma trận ghép [A|b] BAN ĐẦU")

# Thực thi thuật toán
M_res1, sol1, swaps1 = gaussian_eliminate(A1, b1)
display_matrix(M_res1, "Ma trận bậc thang [A|b] sau khi khử Gauss")
print(f"Số lần hoán đổi dòng: {swaps1}")
display_matrix(sol1, "Nghiệm của hệ x", cmap="Greens")

print(">> KIỂM CHỨNG:")
is_correct1 = verify_solution(A1, b1, sol1)
print(f"Kết quả đối chiếu NumPy: {'ĐÚNG' if is_correct1 else 'SAI'}")


print("\nHỆ VÔ SỐ NGHIỆM")
A2 = [[1, 2, 3], 
         [2, 4, 6], 
         [3, 6, 9]]
b2 = [2, 4, 6]

display_matrix(A2, "Ma trận hệ số A")
display_matrix(b2, "Vector vế phải b")
Ab2_init = [row + [b2[i]] for i, row in enumerate(A2)]
display_matrix(Ab2_init, "Ma trận ghép [A|b] BAN ĐẦU")

M_res2, sol2, swaps2 = gaussian_eliminate(A2, b2)
display_matrix(M_res2, "Ma trận bậc thang [A|b] sau khi khử Gauss")
print(f"Số lần hoán đổi dòng: {swaps2}")
print(f"Kết quả: {sol2}")

print("\n>> KIỂM CHỨNG:")
is_correct2 = verify_solution(A2, b2, sol2)
print(f"Kết quả đối chiếu NumPy: {'ĐÚNG' if is_correct2 else 'SAI'}")



print("\nHỆ VÔ NGHIỆM")
A3 = [[1, 2, 3], 
         [2, 4, 6], 
         [3, 6, 9]]
b3 = [2, 4, 100] 
display_matrix(A3, "Ma trận hệ số A")
display_matrix(b3, "Vector vế phải b")
Ab3_init = [row + [b3[i]] for i, row in enumerate(A3)]
display_matrix(Ab3_init, "Ma trận ghép [A|b] BAN ĐẦU")

sol3 = None
try:
    M_res3, sol3, swaps3 = gaussian_eliminate(A3, b3)
    display_matrix(M_res3, "Ma trận sau khi khử Gauss")
except ValueError as e:
    print(f"THÔNG BÁO: {e.args[0]}")
    if len(e.args) > 1:
        M_error = e.args[1]
        display_matrix(M_error, "Ma trận [A|b] sau khi khử Gauss", cmap="Reds")

print("\n>> KIỂM CHỨNG:")
is_correct3 = verify_solution(A3, b3, sol3) 
print(f"Kết quả đối chiếu NumPy: {'ĐÚNG' if is_correct3 else 'SAI'}")

**2. Tính định thức ma trận**

In [ ]:
A_det = [[1, 2, 3], 
         [0, 1, 4], 
         [5, 6, 0]]

det_val = determinant(A_det)

display_matrix(A_det, "Ma trận cần tính định thức")
print(f"\n>> Giá trị định thức: {det_val:.4f}")

is_ok_det = verify_determinant_numpy(A_det, det_val)
print(f">> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_det else 'SAI'}")

**3. Tìm ma trận nghịch đảo** 

In [ ]:
A_inv = [[1, 2, 3], 
              [0, 1, 4], 
              [5, 6, 0]]

inv_res = inverse(A_inv)

display_matrix(A_inv, "Ma trận gốc")
display_matrix(inv_res, "Ma trận nghịch đảo A^-1", cmap="coolwarm")

is_ok_inv = verify_inverse_numpy(A_inv, inv_res)
print(f">> Kiểm chứng NumPy: {'ĐÚNG' if is_ok_inv else 'SAI'}")

**4. Hạng và cơ sở của ma trận** 

In [ ]:

A= [[1, 2, 1, 1],
    [2, 4, 2, 2],
    [3, 6, 3, 3]]

display_matrix(A, "Ma trận A")

rank,r_basis, c_basis, n_basis  = rank_and_basis(A)

print(f"\n>> Hạng của ma trận: {rank}")
display_matrix(r_basis, "Cơ sở không gian dòng ")
display_matrix(c_basis, "Cơ sở không gian cột", cmap="YlGnBu")
if n_basis:
    display_matrix(n_basis, "Cơ sở không gian nghiệm", cmap="Purples")
else:
    print("\n--- Cơ sở không gian nghiệm ---")
    print("   (Hệ chỉ có nghiệm tầm thường, Null Space = {0})")

results = verify_rank_and_basis_numpy(A, rank, r_basis, c_basis, n_basis)
is_all_correct = all(results)
print(f"\n=> Kết quả kiểm chứng tổng thể: {'ĐÚNG' if is_all_correct else 'SAI'}")

# Phần 2: Kiểm tra Test Case

**1. Test - Back Substitution** 

In [ ]:
print(f"\n{AutoTestReporter.COLOR_WARNING}{AutoTestReporter.STYLE_BOLD}KIỂM THỬ THẾ NGƯỢC (BACK SUBSTITUTION){AutoTestReporter.STYLE_RESET}")


print("\nDanh sách các ma trận được sử dụng để kiểm thử:")
df_back_substitution_tests = pd.DataFrame(BACK_SUBSTITUTION_TEST_CASES)
columns_to_show = ['name', 'U', 'c'] 
valid_columns = [col for col in columns_to_show if col in df_back_substitution_tests.columns]

display(df_back_substitution_tests[valid_columns].style.set_properties(**{'text-align': 'left', 'border': '1px solid #ddd'}))

print("\nTiến hành chạy tự động:")
verify_test_back_substitution(BACK_SUBSTITUTION_TEST_CASES)

**2. Test - Gaussian Elimination + Back Substitution** 

In [ ]:
print(f"\n{AutoTestReporter.COLOR_WARNING}{AutoTestReporter.STYLE_BOLD}GIẢI HỆ PHƯƠNG TRÌNH{AutoTestReporter.STYLE_RESET}")


print("\nDanh sách các ma trận được sử dụng để kiểm thử:")
df_gauss_tests = pd.DataFrame(GAUSSIAN_ELIMINATE_TEST_CASES)
columns_to_show = ['name', 'A', 'b'] 
valid_columns = [col for col in columns_to_show if col in df_gauss_tests.columns]

display(df_gauss_tests[valid_columns].style.set_properties(**{'text-align': 'left', 'border': '1px solid #ddd'}))

print("\nTiến hành chạy tự động:")
verify_test_gaussian_eliminate(GAUSSIAN_ELIMINATE_TEST_CASES)

**3. Test - Determinant** 

In [ ]:
print(f"\n{AutoTestReporter.COLOR_WARNING}{AutoTestReporter.STYLE_BOLD}ĐỊNH THỨC{AutoTestReporter.STYLE_RESET}")


print("\nDanh sách các ma trận được sử dụng để kiểm thử:")
df_determinant_tests = pd.DataFrame(DETERMINANT_TEST_CASES)
columns_to_show = ['name', 'A'] 
valid_columns = [col for col in columns_to_show if col in df_determinant_tests.columns]

display(df_determinant_tests[valid_columns].style.set_properties(**{'text-align': 'left', 'border': '1px solid #ddd'}))

print("\nTiến hành chạy tự động:")
verify_test_determinant(DETERMINANT_TEST_CASES)

**4. Test - Inverse** 

In [ ]:
print(f"\n{AutoTestReporter.COLOR_WARNING}{AutoTestReporter.STYLE_BOLD}NGHỊCH ĐẢO{AutoTestReporter.STYLE_RESET}")


print("\nDanh sách các ma trận được sử dụng để kiểm thử:")
df_inverse_tests = pd.DataFrame(INVERSE_TEST_CASES)
columns_to_show = ['name', 'input'] 
valid_columns = [col for col in columns_to_show if col in df_inverse_tests.columns]

display(df_inverse_tests[valid_columns].style.set_properties(**{'text-align': 'left', 'border': '1px solid #ddd'}))

print("\nTiến hành chạy tự động:")
verify_test_inverse(INVERSE_TEST_CASES)

**5. Test - Rank and Basis** 

In [ ]:
print(f"\n{AutoTestReporter.COLOR_WARNING}{AutoTestReporter.STYLE_BOLD}HẠNG VÀ CƠ SỞ{AutoTestReporter.STYLE_RESET}")


print("\nDanh sách các ma trận được sử dụng để kiểm thử:")
df_rank_and_basis_tests = pd.DataFrame(RANK_BASIS_TEST_CASES)
columns_to_show = ['name', 'input'] 
valid_columns = [col for col in columns_to_show if col in df_rank_and_basis_tests.columns]

display(df_rank_and_basis_tests[valid_columns].style.set_properties(**{'text-align': 'left', 'border': '1px solid #ddd'}))

print("\nTiến hành chạy tự động:")
verify_test_rank_and_basis(RANK_BASIS_TEST_CASES)

**6. Test - Verify Solution** 

In [ ]:
print(f"\n{AutoTestReporter.COLOR_WARNING}{AutoTestReporter.STYLE_BOLD}HÀM KIỂM CHỨNG CHO GAUSS{AutoTestReporter.STYLE_RESET}")


print("\nDanh sách các ma trận được sử dụng để kiểm thử:")
df_verify_solution_tests = pd.DataFrame(VERIFY_SOLUTION_TEST_CASES)
columns_to_show = ['name', 'A'] 
valid_columns = [col for col in columns_to_show if col in df_verify_solution_tests.columns]

display(df_verify_solution_tests[valid_columns].style.set_properties(**{'text-align': 'left', 'border': '1px solid #ddd'}))

print("\nTiến hành chạy tự động:")
verify_test_verify_solution(VERIFY_SOLUTION_TEST_CASES)